# Text Vectorization

Converting text to numerical representations for machine learning.

## Learning Objectives

- Understand Bag of Words (BoW)
- Implement TF-IDF vectorization
- Explore word embeddings
- Compare vectorization approaches

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD

plt.style.use('seaborn-v0_8-whitegrid')
print("Libraries loaded!")

## 1. Sample Corpus

In [ ]:
# Sample documents
corpus = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",
    "Natural language processing deals with text and speech.",
    "Machine learning models learn patterns from data.",
    "Neural networks are inspired by the human brain."
]

for i, doc in enumerate(corpus):
    print(f"Doc {i+1}: {doc}")

## 2. Bag of Words (BoW)

In [ ]:
# Manual BoW implementation
def build_vocabulary(documents):
    """Build vocabulary from documents."""
    vocab = set()
    for doc in documents:
        words = doc.lower().split()
        vocab.update(words)
    return sorted(vocab)

def bow_vectorize(document, vocabulary):
    """Convert document to BoW vector."""
    words = document.lower().split()
    word_counts = Counter(words)
    return [word_counts.get(word, 0) for word in vocabulary]

vocab = build_vocabulary(corpus)
print(f"Vocabulary ({len(vocab)} words): {vocab[:10]}...\n")

# Vectorize first document
vec = bow_vectorize(corpus[0], vocab)
print(f"Document: {corpus[0]}")
print(f"Vector length: {len(vec)}")
print(f"Non-zero entries: {sum(1 for v in vec if v > 0)}")

In [ ]:
# Sklearn CountVectorizer
count_vec = CountVectorizer()
bow_matrix = count_vec.fit_transform(corpus)

print(f"BoW Matrix shape: {bow_matrix.shape}")
print(f"Vocabulary size: {len(count_vec.vocabulary_)}\n")

# Show as DataFrame
feature_names = count_vec.get_feature_names_out()
bow_df = pd.DataFrame(
    bow_matrix.toarray(),
    columns=feature_names,
    index=[f'Doc {i+1}' for i in range(len(corpus))]
)
print(bow_df)

In [ ]:
# Customizing CountVectorizer
custom_vec = CountVectorizer(
    lowercase=True,
    stop_words='english',
    min_df=1,  # Minimum document frequency
    max_df=0.9,  # Maximum document frequency (fraction)
    ngram_range=(1, 2)  # Unigrams and bigrams
)

custom_matrix = custom_vec.fit_transform(corpus)
print(f"Custom BoW shape: {custom_matrix.shape}")
print(f"\nFeatures: {custom_vec.get_feature_names_out()[:15]}...")

## 3. TF-IDF Vectorization

In [ ]:
# TF-IDF explanation
def compute_tf(document):
    """Compute term frequency."""
    words = document.lower().split()
    word_counts = Counter(words)
    total_words = len(words)
    return {word: count/total_words for word, count in word_counts.items()}

def compute_idf(documents, vocab):
    """Compute inverse document frequency."""
    n_docs = len(documents)
    idf = {}
    for word in vocab:
        doc_count = sum(1 for doc in documents if word in doc.lower())
        idf[word] = np.log(n_docs / (doc_count + 1)) + 1
    return idf

# Demo
doc = corpus[0]
tf = compute_tf(doc)
idf = compute_idf(corpus, tf.keys())

print(f"Document: {doc}\n")
print(f"{'Word':<15} {'TF':>8} {'IDF':>8} {'TF-IDF':>8}")
print("-" * 42)
for word in list(tf.keys())[:5]:
    tfidf = tf[word] * idf[word]
    print(f"{word:<15} {tf[word]:>8.3f} {idf[word]:>8.3f} {tfidf:>8.3f}")

In [ ]:
# Sklearn TfidfVectorizer
tfidf_vec = TfidfVectorizer()
tfidf_matrix = tfidf_vec.fit_transform(corpus)

print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}\n")

# Show as DataFrame
tfidf_df = pd.DataFrame(
    tfidf_matrix.toarray().round(3),
    columns=tfidf_vec.get_feature_names_out(),
    index=[f'Doc {i+1}' for i in range(len(corpus))]
)
print(tfidf_df)

In [ ]:
# Compare important words per document
print("=== Top 3 Words by TF-IDF Score ===")
for i, doc in enumerate(corpus):
    row = tfidf_matrix[i].toarray().flatten()
    top_indices = row.argsort()[-3:][::-1]
    top_words = [(tfidf_vec.get_feature_names_out()[idx], row[idx]) 
                 for idx in top_indices]
    print(f"\nDoc {i+1}: {doc[:40]}...")
    for word, score in top_words:
        print(f"  {word}: {score:.3f}")

## 4. Document Similarity

In [ ]:
# Cosine similarity
similarity_matrix = cosine_similarity(tfidf_matrix)

sim_df = pd.DataFrame(
    similarity_matrix.round(3),
    columns=[f'Doc {i+1}' for i in range(len(corpus))],
    index=[f'Doc {i+1}' for i in range(len(corpus))]
)
print("=== Document Similarity Matrix ===")
print(sim_df)

In [ ]:
# Visualize similarity
plt.figure(figsize=(8, 6))
plt.imshow(similarity_matrix, cmap='YlOrRd', aspect='auto')
plt.colorbar(label='Cosine Similarity')
plt.xticks(range(len(corpus)), [f'Doc {i+1}' for i in range(len(corpus))])
plt.yticks(range(len(corpus)), [f'Doc {i+1}' for i in range(len(corpus))])
plt.title('Document Similarity Heatmap')

# Add values
for i in range(len(corpus)):
    for j in range(len(corpus)):
        plt.text(j, i, f'{similarity_matrix[i, j]:.2f}',
                ha='center', va='center', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# Find similar documents
def find_similar(query, vectorizer, doc_matrix, documents, top_n=3):
    """Find most similar documents to a query."""
    query_vec = vectorizer.transform([query])
    similarities = cosine_similarity(query_vec, doc_matrix).flatten()
    
    top_indices = similarities.argsort()[-top_n:][::-1]
    results = [(documents[i], similarities[i]) for i in top_indices]
    return results

# Query
query = "neural network learning"
results = find_similar(query, tfidf_vec, tfidf_matrix, corpus)

print(f"Query: '{query}'\n")
print("Most similar documents:")
for doc, score in results:
    print(f"  [{score:.3f}] {doc}")

## 5. Dimensionality Reduction (LSA)

In [ ]:
# Latent Semantic Analysis
n_components = 2
svd = TruncatedSVD(n_components=n_components, random_state=42)
lsa_matrix = svd.fit_transform(tfidf_matrix)

print(f"Original shape: {tfidf_matrix.shape}")
print(f"LSA shape: {lsa_matrix.shape}")
print(f"Explained variance: {svd.explained_variance_ratio_.sum():.2%}")

In [ ]:
# Visualize in 2D
plt.figure(figsize=(10, 8))
plt.scatter(lsa_matrix[:, 0], lsa_matrix[:, 1], s=100, c='steelblue')

for i, doc in enumerate(corpus):
    plt.annotate(
        f'Doc {i+1}',
        (lsa_matrix[i, 0], lsa_matrix[i, 1]),
        fontsize=10,
        xytext=(5, 5),
        textcoords='offset points'
    )

plt.xlabel('LSA Component 1')
plt.ylabel('LSA Component 2')
plt.title('Document Embeddings (LSA)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. One-Hot Encoding for Words

In [ ]:
# One-hot encoding
def one_hot_encode(word, vocabulary):
    """Create one-hot vector for a word."""
    vector = [0] * len(vocabulary)
    if word.lower() in vocabulary:
        idx = vocabulary.index(word.lower())
        vector[idx] = 1
    return vector

# Build vocabulary
simple_vocab = ['deep', 'learning', 'machine', 'neural', 'network']

print(f"Vocabulary: {simple_vocab}\n")
print("One-hot encodings:")
for word in simple_vocab:
    vec = one_hot_encode(word, simple_vocab)
    print(f"  {word:<10}: {vec}")

## 7. Comparing Vectorization Methods

In [ ]:
# Compare methods
methods = {
    'BoW': CountVectorizer(),
    'TF-IDF': TfidfVectorizer(),
    'BoW + Bigrams': CountVectorizer(ngram_range=(1, 2)),
    'TF-IDF + Stopwords': TfidfVectorizer(stop_words='english')
}

print("=== Vectorization Method Comparison ===")
print(f"{'Method':<25} {'Features':>10} {'Sparsity':>12}")
print("-" * 50)

for name, vec in methods.items():
    matrix = vec.fit_transform(corpus)
    n_features = matrix.shape[1]
    sparsity = 1 - (matrix.nnz / (matrix.shape[0] * matrix.shape[1]))
    print(f"{name:<25} {n_features:>10} {sparsity:>11.1%}")

## 8. Practical Application: Text Classification Setup

In [ ]:
# Larger dataset for classification
texts = [
    ("Stock prices rose sharply today on Wall Street", "finance"),
    ("The football team won the championship game", "sports"),
    ("New machine learning algorithm improves predictions", "tech"),
    ("Interest rates expected to increase next quarter", "finance"),
    ("Tennis player wins Grand Slam tournament", "sports"),
    ("Deep learning models beat human performance", "tech"),
    ("Market volatility concerns investors", "finance"),
    ("Basketball team trades star player", "sports"),
]

documents, labels = zip(*texts)

# Vectorize
vectorizer = TfidfVectorizer(stop_words='english')
X = vectorizer.fit_transform(documents)

print(f"Feature matrix shape: {X.shape}")
print(f"Labels: {set(labels)}")
print(f"\nFeatures: {vectorizer.get_feature_names_out()}")

## 9. Key Takeaways

| Method | Pros | Cons |
|--------|------|------|
| **BoW** | Simple, interpretable | Loses word order |
| **TF-IDF** | Weights important words | Still sparse |
| **N-grams** | Captures phrases | Exploding dimensionality |
| **LSA** | Reduces dimensions | Less interpretable |

### When to Use What
- **BoW**: Simple classification, baseline models
- **TF-IDF**: Document retrieval, keyword extraction
- **LSA**: Topic modeling, semantic similarity
- **Word Embeddings**: (Next notebook) - Semantic tasks